# 🇧🇩 Bangladesh Bar Council MCQ Generator
## RAG-based Legal MCQ Generation System

**Source:** Ministry of Law, Justice and Parliamentary Affairs, Bangladesh (bdlaws.minlaw.gov.bd)

**Laws Covered:**
1. দণ্ডবিধি (The Penal Code, 1860)
2. ফৌজদারি কার্যবিধি (Code of Criminal Procedure, 1898)
3. দেওয়ানি কার্যবিধি (Code of Civil Procedure, 1908)
4. সাক্ষ্য আইন (The Evidence Act, 1872)
5. তামাদি আইন (The Limitation Act, 1908)
6. সুনির্দিষ্ট প্রতিকার আইন (The Specific Relief Act, 1877)

**Total RAG Chunks:** 3,268 (1,634 EN + 1,634 BN)

## Step 1: Install Dependencies

In [ ]:
!pip install -q chromadb sentence-transformers openai langchain langchain-community tiktoken

## Step 2: Upload Data
Upload the `data/target_laws/` folder or the generated `rag_chunks.jsonl` file.

In [ ]:
from google.colab import files
import os

# Option 1: Upload rag_chunks.jsonl directly
# uploaded = files.upload()

# Option 2: Mount Google Drive (if data is there)
from google.colab import drive
drive.mount('/content/drive')

# Set your data path
DATA_PATH = '/content/drive/MyDrive/bdlaws-6laws-rag/'
# Or if uploaded directly:
# DATA_PATH = '/content/'

## Step 3: Load RAG Chunks & Create Vector Store

In [ ]:
import json
import chromadb
from sentence_transformers import SentenceTransformer

# Load chunks
chunks = []
with open(os.path.join(DATA_PATH, 'rag_chunks.jsonl'), 'r', encoding='utf-8') as f:
    for line in f:
        chunks.append(json.loads(line))

print(f'Loaded {len(chunks)} RAG chunks')
print(f'Priority chunks: {sum(1 for c in chunks if c["is_priority"])}')

In [ ]:
# Create embeddings using multilingual model (supports Bengali)
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Prepare texts for embedding
texts = []
ids = []
metadatas = []

for chunk in chunks:
    # Combine EN + BN content for better retrieval
    text = f"{chunk['law_title_bn']} - {chunk['title']}\n{chunk['content_en']}"
    texts.append(text)
    ids.append(chunk['id'])
    metadatas.append({
        'law_key': chunk['law_key'],
        'law_title': chunk['law_title_bn'],
        'section_num': chunk['section_num'],
        'title': chunk['title'],
        'is_priority': str(chunk['is_priority']),
    })

print(f'Encoding {len(texts)} chunks...')
embeddings = model.encode(texts, show_progress_bar=True, batch_size=64)
print(f'Embeddings shape: {embeddings.shape}')

In [ ]:
# Create ChromaDB collection
client = chromadb.Client()
collection = client.create_collection(
    name='bangladesh_laws',
    metadata={'description': 'Bangladesh 6 Laws for Bar Council MCQ Generation'}
)

# Add to collection in batches
batch_size = 100
for i in range(0, len(texts), batch_size):
    end = min(i + batch_size, len(texts))
    collection.add(
        documents=texts[i:end],
        embeddings=embeddings[i:end].tolist(),
        ids=ids[i:end],
        metadatas=metadatas[i:end],
    )

print(f'✅ Vector store created with {collection.count()} documents')

## Step 4: MCQ Generation with RAG

In [ ]:
import openai

# Set your API key
openai.api_key = 'YOUR_API_KEY_HERE'  # Replace with your key

SYSTEM_PROMPT = """You are a Bangladesh Bar Council legal examiner AI specialized in generating MCQs for the Advocates' Enrolment Examination.

RULES:
1. Generate MCQs ONLY from authentic Bangladesh law sections provided in context
2. Use professional Bengali legal terminology
3. Each MCQ must have 4 options (ক, খ, গ, ঘ)
4. Provide correct answer with section-based explanation
5. NEVER invent sections or legal principles
6. Test technical understanding, exceptions, and procedural knowledge

OUTPUT FORMAT:
### প্রশ্ন:
(Bengali question)

### অপশন:
ক. (option)
খ. (option)
গ. (option)
ঘ. (option)

### সঠিক উত্তর:
(correct option letter)

### ব্যাখ্যা:
(Section reference + explanation in Bengali)
"""


def retrieve_context(query, n_results=5, law_filter=None):
    """Retrieve relevant law sections for MCQ generation."""
    where_filter = None
    if law_filter:
        where_filter = {'law_key': law_filter}
    
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        where=where_filter,
    )
    return results['documents'][0]


def generate_mcq(topic, law_filter=None, difficulty='medium', n_questions=5):
    """Generate MCQs using RAG."""
    # Retrieve relevant sections
    context_docs = retrieve_context(topic, n_results=10, law_filter=law_filter)
    context = '\n\n---\n\n'.join(context_docs)
    
    user_prompt = f"""Based on the following authentic Bangladesh law sections, generate {n_questions} Bar Council MCQs.

Topic: {topic}
Difficulty: {difficulty}
Language: Bengali (professional legal Bengali)

CONTEXT (Authentic Law Sections):
{context}

Generate {n_questions} MCQs following the exact format specified. Each question must directly reference the sections provided."""
    
    response = openai.chat.completions.create(
        model='gpt-4o',
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': user_prompt},
        ],
        temperature=0.7,
        max_tokens=4000,
    )
    
    return response.choices[0].message.content


print('✅ MCQ Generator ready!')

## Step 5: Generate MCQs

In [ ]:
# Example: Generate CrPC MCQs on Bail
mcqs = generate_mcq(
    topic='জামিন (Bail) - ফৌজদারি কার্যবিধি',
    law_filter='crpc',
    difficulty='medium',
    n_questions=5
)
print(mcqs)

In [ ]:
# Example: Generate Evidence Act MCQs
mcqs = generate_mcq(
    topic='স্বীকারোক্তি ও মৃত্যুকালীন ঘোষণা (Confession and Dying Declaration)',
    law_filter='evidence_act',
    difficulty='hard',
    n_questions=5
)
print(mcqs)

In [ ]:
# Example: Generate Penal Code MCQs on General Exceptions
mcqs = generate_mcq(
    topic='সাধারণ ব্যতিক্রম (General Exceptions) - Right of Private Defence',
    law_filter='penal_code',
    difficulty='hard',
    n_questions=5
)
print(mcqs)

## Step 6: Batch Generation (Full Exam Set)

In [ ]:
# Generate a full mock exam set following Bar Council weight distribution
EXAM_TOPICS = [
    # CrPC (25%)
    ('ফৌজদারি কার্যবিধি - আমলযোগ্য ও অ-আমলযোগ্য অপরাধ', 'crpc', 'medium', 3),
    ('ফৌজদারি কার্যবিধি - জামিন ও গ্রেফতার', 'crpc', 'medium', 3),
    ('ফৌজদারি কার্যবিধি - বিচার পদ্ধতি (সমন ও ওয়ারেন্ট মামলা)', 'crpc', 'hard', 3),
    ('ফৌজদারি কার্যবিধি - ম্যাজিস্ট্রেটের ক্ষমতা', 'crpc', 'medium', 3),
    ('ফৌজদারি কার্যবিধি - আপিল ও রিভিশন', 'crpc', 'hard', 3),
    
    # CPC (25%)
    ('দেওয়ানি কার্যবিধি - Res Judicata ও Res Sub Judice', 'cpc', 'hard', 3),
    ('দেওয়ানি কার্যবিধি - অস্থায়ী নিষেধাজ্ঞা', 'cpc', 'medium', 3),
    ('দেওয়ানি কার্যবিধি - ডিক্রি ও আদেশ', 'cpc', 'medium', 3),
    ('দেওয়ানি কার্যবিধি - আপিল, রিভিশন ও রিভিউ', 'cpc', 'hard', 3),
    ('দেওয়ানি কার্যবিধি - নিষ্পত্তি (Execution)', 'cpc', 'medium', 3),
    
    # Evidence Act (15%)
    ('সাক্ষ্য আইন - প্রমাণের ভার (Burden of Proof)', 'evidence_act', 'medium', 3),
    ('সাক্ষ্য আইন - অনুমান (Presumptions)', 'evidence_act', 'hard', 2),
    ('সাক্ষ্য আইন - জেরা ও দলিলী সাক্ষ্য', 'evidence_act', 'medium', 3),
    ('সাক্ষ্য আইন - স্বীকারোক্তি', 'evidence_act', 'hard', 2),
    
    # Penal Code (15%)
    ('দণ্ডবিধি - সাধারণ ব্যতিক্রম', 'penal_code', 'hard', 3),
    ('দণ্ডবিধি - দুষ্প্রেরণা ও সাধারণ অভিপ্রায়', 'penal_code', 'medium', 2),
    ('দণ্ডবিধি - চুরি, দস্যুতা, প্রতারণা', 'penal_code', 'medium', 3),
    ('দণ্ডবিধি - হত্যা ও আঘাত', 'penal_code', 'hard', 2),
    
    # Limitation Act (8%)
    ('তামাদি আইন - তামাদি গণনা ও ব্যতিক্রম', 'limitation_act', 'medium', 3),
    ('তামাদি আইন - আইনগত অক্ষমতা ও স্বীকৃতি', 'limitation_act', 'hard', 2),
    
    # Specific Relief Act (7%)
    ('সুনির্দিষ্ট প্রতিকার আইন - চুক্তির সুনির্দিষ্ট কার্যসম্পাদন', 'specific_relief', 'medium', 2),
    ('সুনির্দিষ্ট প্রতিকার আইন - নিষেধাজ্ঞা ও ঘোষণামূলক ডিক্রি', 'specific_relief', 'medium', 3),
]

total_questions = sum(t[3] for t in EXAM_TOPICS)
print(f'Generating {total_questions} MCQs across {len(EXAM_TOPICS)} topics...')
print(f'This will take several minutes.\n')

all_mcqs = []
for topic, law, difficulty, n_q in EXAM_TOPICS:
    print(f'  Generating: {topic} ({n_q} questions, {difficulty})...')
    try:
        result = generate_mcq(topic, law_filter=law, difficulty=difficulty, n_questions=n_q)
        all_mcqs.append(f'\n## {topic}\n\n{result}')
    except Exception as e:
        print(f'    Error: {e}')

# Save full exam
with open('bar_council_mock_exam.md', 'w', encoding='utf-8') as f:
    f.write('# বাংলাদেশ বার কাউন্সিল পরীক্ষা - মক টেস্ট\n\n')
    f.write('\n'.join(all_mcqs))

print(f'\n✅ Mock exam saved to bar_council_mock_exam.md')

## Alternative: Use with Gemini (Free)

In [ ]:
# If using Google Gemini instead of OpenAI
# !pip install -q google-generativeai

# import google.generativeai as genai
# genai.configure(api_key='YOUR_GEMINI_API_KEY')
# model = genai.GenerativeModel('gemini-1.5-pro')
#
# def generate_mcq_gemini(topic, law_filter=None, difficulty='medium', n_questions=5):
#     context_docs = retrieve_context(topic, n_results=10, law_filter=law_filter)
#     context = '\n\n---\n\n'.join(context_docs)
#     
#     prompt = f"""{SYSTEM_PROMPT}
#
# Based on the following authentic Bangladesh law sections, generate {n_questions} Bar Council MCQs.
# Topic: {topic}
# Difficulty: {difficulty}
#
# CONTEXT:
# {context}
# """
#     response = model.generate_content(prompt)
#     return response.text